In [ ]:
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
)

In [ ]:
df = pd.read_csv("../data/raw/bank-full.csv", sep=";")
df.shape

In [ ]:
before_len = len(df)
df = df[~((df["poutcome"] == "unknown") & (df["pdays"] != -1))]
after_len = len(df)
print(f"length before: {before_len}, after: {after_len}")

In [ ]:
df.drop(["pdays", "day", "duration"], axis=1, inplace=True)
df.info()

In [ ]:
df["y"] = (df["y"] == "yes").astype(int)
print(df["y"].value_counts())

In [ ]:
X = df.drop("y", axis=1)
y = df["y"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
cat_cols = X_train.select_dtypes(include="str").columns

In [ ]:
ct = ColumnTransformer(
    [
        ("Categorical columns", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("Balance", PowerTransformer(method="yeo-johnson"), ["balance"]),
        (
            "age",
            KBinsDiscretizer(n_bins=5, strategy="quantile", encode="onehot"),
            ["age"],
        ),
    ],
    remainder="passthrough",
)

In [ ]:
ct.fit(X_train)
X_train_t = ct.transform(X_train)
X_test_t = ct.transform(X_test)
X_train_t.shape

In [ ]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [ ]:
joblib.dump(ct, "../models/preprocessor.joblib")